# Train the "Hai Nemo" and "Tolong" wake word models

This notebook trains the two [openWakeWord](https://github.com/dscripka/openWakeWord) models AIForUs
uses when `listening.wake_engine` is `openwakeword`:

| File | Listens for | Config key |
|---|---|---|
| `hai_nemo.onnx` | "Hai Nemo", "Halo Nemo", "Nemo" | `listening.openwakeword_wake_model` |
| `tolong.onnx` | "Tolong", "Darurat" (emergency, no name needed) | `listening.openwakeword_emergency_model` |

**Before you start**

1. *Runtime → Change runtime type → T4 GPU*. Without a GPU, generating the clips takes many hours.
2. Choose the settings in the next cell, then *Runtime → Run all*.
3. Expect roughly 1.5-3 hours with `SIZE = "small"` for both models (most of it downloading data and
   generating clips) - longer with `"full"`. If Colab disconnects, run all again: finished steps are skipped.

**Privacy.** By default nothing of yours is uploaded: the models learn from synthetic voices only.
Real recordings of your voice (`python -m aiforus --record-wake-samples`) make the models much better
at Indonesian pronunciation, but using them means uploading them to Google's servers. That is your
choice: set `USE_REAL_RECORDINGS = True` only for recordings of people who agreed, and delete the
runtime afterwards (*Runtime → Disconnect and delete runtime*). To keep recordings on your own
computer instead, run the same steps in WSL2 (see `tools/wakeword/README.md`).

In [ ]:
# "small" = first try (faster), "full" = the size openWakeWord recommends for daily use
SIZE = "small"
# Which models to train: "hai_nemo" (wake word) and/or "tolong" (emergency)
MODELS = ["hai_nemo", "tolong"]
# True = upload wake_samples.zip with your real recordings (see the privacy note above)
USE_REAL_RECORDINGS = False
# Hours of music used as background noise ("small": 1, "full": 3 or more)
MUSIC_HOURS = 1 if SIZE == "small" else 3

## 1. Install

In [ ]:
# Install the training tools (about 5 minutes). Versions are pinned: newer ones break training.
!nvidia-smi -L || echo "No GPU - choose Runtime > Change runtime type > T4 GPU"
!test -d openwakeword || git clone -q https://github.com/dscripka/openWakeWord openwakeword
!git -C openwakeword checkout -q 368c03716d1e92591906a84949bc477f3a834455
!test -d piper-sample-generator || git clone -q --depth 1 -b v2.0.0 https://github.com/rhasspy/piper-sample-generator
!test -f piper-sample-generator/models/en_US-libritts_r-medium.pt || wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
# torchaudio 2.9+ dropped the audio functions openWakeWord's augmentation uses
!pip install -q "torch==2.8.0" "torchaudio==2.8.0" "torchvision==0.23.0" soundfile
!pip install -q -e ./openwakeword piper-phonemize-fix webrtcvad-wheels "mutagen>=1.46,<2" "torchinfo>=1.8,<2" "torchmetrics>=0.11.4,<1" "speechbrain>=0.5.14,<1" "audiomentations>=0.30,<1" "torch-audiomentations>=0.11,<1" "acoustics>=0.2.6,<1" "pronouncing>=0.2,<1" "datasets>=2.14.4,<3" "deep-phonemizer==0.0.19" "scipy<1.17" "setuptools<81" pyyaml onnx onnxruntime

import hashlib, os, subprocess, sys
from pathlib import Path

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

# The Piper checkpoint is loaded with pickle (see TRAIN_ENV in the helper), so it must be the exact file.
assert sha256(f"piper-sample-generator/models/en_US-libritts_r-medium.pt") == "e95ee53770bf598c354a6e6dbfc95ccb259aeeb501d35a86be8a767429ab0ff6", "unexpected Piper checkpoint - delete it and run again"

resources = Path("openwakeword/openwakeword/resources/models")
resources.mkdir(parents=True, exist_ok=True)
for name, digest in {'melspectrogram.onnx': 'ba2b0e0f8b7b875369a2c89cb13360ff53bac436f2895cced9f479fa65eb176f', 'embedding_model.onnx': '70d164290c1d095d1d4ee149bc5e00543250a7316b59f31d056cff7bd3075c1f'}.items():
    target = resources / name
    if not target.exists():
        subprocess.run(["wget", "-q", "-O", str(target), f"https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/{name}"], check=True)
    assert sha256(target) == digest, f"unexpected {name}"
sys.path.insert(0, "openwakeword")  # the editable install is only seen by a restarted kernel
print("Tools ready.")

## 2. Helper
The AIForUs helper (`tools/wakeword/nemo_wakeword.py`): phrases, real recordings, checks.

In [ ]:
%%writefile nemo_wakeword.py
"""Helpers for training the "Hai Nemo" and "Tolong" wake word models.

This file runs on Google Colab (openWakeWord training is Linux-only); the
notebook ``train_nemo.ipynb`` writes it to disk with ``%%writefile`` and the
tests in ``tests/test_wakeword_tools.py`` import it here. It only needs numpy
and scipy, plus openwakeword for :func:`score_wavs`.

What it adds to the standard openWakeWord recipe:

* **presets** for the two AIForUs models: phrases, phonetic spellings that make
  the English training voices sound closer to Indonesian, and look-alike words
  that must *not* trigger;
* **real recordings** from ``python -m aiforus --record-wake-samples``: trimmed,
  resampled to 16 kHz and mixed into the synthetic training clips, with a part
  held back to check the finished model on real voices.
"""

from __future__ import annotations

import random
import shutil
import wave
import zipfile
from dataclasses import dataclass
from math import gcd
from pathlib import Path

import numpy as np

SAMPLE_RATE = 16000


@dataclass(frozen=True)
class Preset:
    model_name: str
    target_phrase: tuple[str, ...]
    negative_phrases: tuple[str, ...]
    sample_folders: tuple[str, ...]  # data/wake_samples/<folder> from the recorder


PRESETS: dict[str, Preset] = {
    # The model file name is what AIForUs loads: listening.openwakeword_wake_model.
    "hai_nemo": Preset(
        model_name="hai_nemo",
        target_phrase=(
            "hi nemo",
            "hey nemo",
            "hello nemo",
            "nemo",
            # Phonetic spellings: Indonesian "Nemo" is "neh-moh", not "nee-moh".
            "hi neh moh",
            "hah lo neh moh",
            "neh moh",
        ),
        negative_phrases=(
            "demo",
            "memo",
            "emo",
            "nemu",
            "nenek",
            "lemon",
            "nomor",
            "nama",
            "enam",
            "hi demo",
            "hello memo",
            "hey emo",
            "hai lemon",
            "nanti",
            "ne mau",
        ),
        sample_folders=("hai_nemo", "halo_nemo", "nemo"),
    ),
    "tolong": Preset(
        model_name="tolong",
        target_phrase=(
            "tolong",
            "toh long",
            "toe long",
            "tolong tolong",
            "darurat",
            "dah roo raht",
        ),
        negative_phrases=(
            "tulang",
            "potong",
            "gotong",
            "kolong",
            "dorong",
            "sarung",
            "belong",
            "along",
            "darat",
            "darah",
            "durian",
            "the long",
            "too long",
        ),
        sample_folders=("tolong", "darurat"),
    ),
}

SIZES = {
    # n_samples, n_samples_val, steps. "small" is a first try; "full" is the
    # size the openWakeWord docs recommend for a model you rely on.
    "small": (3000, 1000, 15000),
    "full": (10000, 2000, 50000),
}


def make_config(base: dict, preset: Preset, *, size: str = "small", **paths: object) -> dict:
    """The openWakeWord training config for ``preset``, based on the repo's
    ``examples/custom_model.yml`` (``base``). ``paths`` overrides keys such as
    ``output_dir`` or ``background_paths``."""
    n_samples, n_samples_val, steps = SIZES[size]
    config = dict(base)
    config.update(
        model_name=preset.model_name,
        target_phrase=list(preset.target_phrase),
        custom_negative_phrases=list(preset.negative_phrases),
        n_samples=n_samples,
        n_samples_val=n_samples_val,
        steps=steps,
    )
    config.update(paths)
    return config


# -- real recordings -------------------------------------------------------------


def read_wav(path: Path) -> tuple[np.ndarray, int]:
    """16-bit PCM WAV -> (float32 mono in -1..1, sample rate)."""
    with wave.open(str(path), "rb") as wav:
        if wav.getsampwidth() != 2:
            raise ValueError(f"{path.name}: only 16-bit WAV is supported")
        rate = wav.getframerate()
        channels = wav.getnchannels()
        data = np.frombuffer(wav.readframes(wav.getnframes()), dtype="<i2").astype(np.float32)
    if channels > 1:
        data = data.reshape(-1, channels).mean(axis=1)
    return data / 32768.0, rate


def write_wav(path: Path, audio: np.ndarray, rate: int = SAMPLE_RATE) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    pcm = (np.clip(audio, -1.0, 1.0) * 32767).astype("<i2")
    with wave.open(str(path), "wb") as wav:
        wav.setnchannels(1)
        wav.setsampwidth(2)
        wav.setframerate(rate)
        wav.writeframes(pcm.tobytes())


def resample(audio: np.ndarray, rate: int, target: int = SAMPLE_RATE) -> np.ndarray:
    if rate == target:
        return audio
    from scipy.signal import resample_poly

    g = gcd(rate, target)
    return resample_poly(audio, target // g, rate // g).astype(np.float32)


def trim_silence(
    audio: np.ndarray, rate: int = SAMPLE_RATE, *, frame_s: float = 0.02, margin_s: float = 0.1
) -> np.ndarray:
    """Cut the quiet parts before and after the phrase.

    openWakeWord places each training clip near the *end* of its window, so a
    recording that ends with a second of silence would teach the model that
    the phrase ends a second before the window does. Returns an empty array
    when nothing stands out from the background (a failed recording).
    """
    frame = max(1, int(rate * frame_s))
    n = len(audio) // frame
    if n == 0:
        return audio[:0]
    energy = np.sqrt(np.mean(audio[: n * frame].reshape(n, frame) ** 2, axis=1))
    floor = float(np.percentile(energy, 10))
    peak = float(energy.max())
    if peak < 0.01 or peak < floor * 4:
        return audio[:0]
    threshold = max(floor * 3, peak * 0.1)
    loud = np.flatnonzero(energy > threshold)
    margin = int(rate * margin_s)
    start = max(0, loud[0] * frame - margin)
    end = min(len(audio), (loud[-1] + 1) * frame + margin)
    return audio[start:end]


def unpack_samples(source: Path, dest: Path) -> Path:
    """Unzip ``wake_samples.zip`` (or use a folder as is). Returns the folder
    that contains the phrase folders (hai_nemo/, tolong/, ...)."""
    source = Path(source)
    if source.is_dir():
        root = source
    else:
        dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(source) as archive:
            for member in archive.namelist():
                target = (dest / member).resolve()
                if not target.is_relative_to(dest.resolve()):
                    raise ValueError(f"unsafe path in zip: {member}")
            archive.extractall(dest)
        root = dest
    # The zip may contain "wake_samples/hai_nemo/..." or "hai_nemo/..." directly.
    for candidate in (root, *sorted(p for p in root.iterdir() if p.is_dir())):
        if any((candidate / name).is_dir() for preset in PRESETS.values() for name in preset.sample_folders):
            return candidate
    raise FileNotFoundError(f"no phrase folders (hai_nemo/, tolong/, ...) in {source}")


@dataclass
class RealClips:
    train: list[Path]
    held_out: list[Path]
    skipped: list[Path]


def prepare_real_clips(
    samples_root: Path, preset: Preset, out_dir: Path, *, hold_out: float = 0.2, seed: int = 0
) -> RealClips:
    """Trim and resample the real recordings for ``preset`` into ``out_dir``.

    A share (``hold_out``) is kept out of training so the finished model can
    be checked on voices it has never heard.
    """
    clips: list[Path] = []
    skipped: list[Path] = []
    for folder in preset.sample_folders:
        for path in sorted((samples_root / folder).glob("*.wav")):
            audio, rate = read_wav(path)
            audio = trim_silence(resample(audio, rate))
            if len(audio) < SAMPLE_RATE * 0.2:  # silence or a click
                skipped.append(path)
                continue
            target = out_dir / folder / path.name
            write_wav(target, audio)
            clips.append(target)
    random.Random(seed).shuffle(clips)
    n_held = round(len(clips) * hold_out) if len(clips) >= 5 else 0
    return RealClips(train=clips[n_held:], held_out=clips[:n_held], skipped=skipped)


def add_to_training(
    clips: list[Path], positive_train_dir: Path, *, n_synthetic: int, share: float = 0.2
) -> int:
    """Copy real clips into openWakeWord's ``positive_train`` folder.

    Call this *after* ``train.py --generate_clips`` (otherwise it generates
    fewer synthetic clips) and *before* ``--augment_clips``. Each clip is copied
    several times so real voices make up about ``share`` of the positives; the
    augmentation step gives every copy different noise and room echo.
    Returns the number of files added.
    """
    if not clips:
        return 0
    copies = max(1, round(n_synthetic * share / (1 - share) / len(clips)))
    positive_train_dir.mkdir(parents=True, exist_ok=True)
    added = 0
    for clip in clips:
        for i in range(copies):
            shutil.copyfile(clip, positive_train_dir / f"real_{clip.parent.name}_{clip.stem}_{i:03d}.wav")
            added += 1
    return added


# -- the whole recipe for one model ------------------------------------------------


# PyTorch 2.6+ refuses to unpickle the Piper and DeepPhonemizer checkpoints
# that train.py loads ("Weights only load failed"). This switches the old
# behaviour back for the training steps only; the Piper checkpoint is checked
# against a fixed SHA-256 by the notebook before it is ever loaded.
TRAIN_ENV = {"TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD": "1"}


def run(cmd: list[str], env: dict[str, str] | None = None) -> None:
    """Run a command and stream its output (a notebook does not show the output
    of a plain subprocess). Raises if it fails, so a broken step stops the run."""
    import os
    import subprocess

    print("$", " ".join(cmd), flush=True)
    full_env = {**os.environ, **(env or {})}
    with subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=full_env
    ) as proc:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="", flush=True)
    if proc.returncode:
        raise RuntimeError(f"step failed (exit code {proc.returncode}): {' '.join(cmd)}")


# train.py declares --convert_to_tflite with default="False" - a non-empty
# string, so it always tries the TFLite conversion and crashes without the old
# TensorFlow 2.8 stack (which no longer installs on Colab). AIForUs only needs
# the ONNX model, so the check is made strict like the other flags.
_TFLITE_CHECK = "if args.convert_to_tflite:"
_TFLITE_FIXED = "if args.convert_to_tflite is True:"


def patch_train_script(train_script: Path) -> None:
    text = Path(train_script).read_text(encoding="utf-8")
    if _TFLITE_CHECK in text:
        Path(train_script).write_text(text.replace(_TFLITE_CHECK, _TFLITE_FIXED), encoding="utf-8")
    elif _TFLITE_FIXED not in text:
        raise RuntimeError(f"{train_script} changed upstream; check its --convert_to_tflite handling")


@dataclass
class TrainResult:
    model_path: Path
    real_added: int
    held_out: list[Path]
    skipped: list[Path]


def train_model(
    preset: Preset,
    *,
    base_config: dict,
    train_script: Path,
    work_dir: Path,
    size: str = "small",
    samples_root: Path | None = None,
    overrides: dict | None = None,
    runner=run,
) -> TrainResult:
    """openWakeWord's three steps, with the real recordings slipped in between
    generating and augmenting. Safe to re-run after a disconnect: clips that
    already exist are kept, and augmentation is redone unless it finished."""
    import sys

    import yaml

    patch_train_script(train_script)
    config = make_config(base_config, preset, size=size, **(overrides or {}))
    config_path = work_dir / f"{preset.model_name}.yaml"
    config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    base = [sys.executable, str(train_script), "--training_config", str(config_path)]

    runner([*base, "--generate_clips"], TRAIN_ENV)

    real = RealClips([], [], [])
    added = 0
    if samples_root is not None:
        real = prepare_real_clips(samples_root, preset, work_dir / "real_clips" / preset.model_name)
        positive_train = Path(config["output_dir"]) / preset.model_name / "positive_train"
        n_synthetic = sum(1 for p in positive_train.glob("*.wav") if not p.name.startswith("real_"))
        added = sum(1 for _ in positive_train.glob("real_*.wav"))
        if not added:  # a re-run must not add them twice
            added = add_to_training(real.train, positive_train, n_synthetic=n_synthetic)
        print(
            f"Real recordings for '{preset.model_name}': {len(real.train)} for training "
            f"({added} copies), {len(real.held_out)} kept back for checking, "
            f"{len(real.skipped)} skipped (silent or failed)."
        )

    # train.py skips augmentation when the feature files exist - even half-written
    # ones from a run that was cut off. So it only counts as done once it finished.
    augmented = Path(config["output_dir"]) / preset.model_name / "augmented.done"
    if not augmented.exists():
        runner([*base, "--augment_clips", "--overwrite"], TRAIN_ENV)
        augmented.touch()
    runner([*base, "--train_model"], TRAIN_ENV)
    model_path = Path(config["output_dir"]) / f"{preset.model_name}.onnx"
    if not model_path.exists():
        raise FileNotFoundError(f"training finished but {model_path} is missing")
    return TrainResult(model_path, added, real.held_out, real.skipped)


# -- checking the finished model ---------------------------------------------------


def score_wavs(model_path: Path, wavs: list[Path], *, pad_s: float = 1.0) -> dict[str, float]:
    """Highest score the model gives each clip (padded with silence, fed in
    80 ms frames like AIForUs does)."""
    from openwakeword.model import Model

    model = Model(wakeword_models=[str(model_path)], inference_framework="onnx")
    key = Path(model_path).stem
    pad = np.zeros(int(SAMPLE_RATE * pad_s), dtype=np.int16)
    scores: dict[str, float] = {}
    for path in wavs:
        audio, rate = read_wav(path)
        pcm = (np.clip(resample(audio, rate), -1, 1) * 32767).astype(np.int16)
        pcm = np.concatenate([pad, pcm, pad])
        model.reset()
        best = 0.0
        for start in range(0, len(pcm) - 1279, 1280):
            best = max(best, float(model.predict(pcm[start : start + 1280])[key]))
        scores[path.name] = best
    return scores

In [ ]:
import importlib, nemo_wakeword as nw
importlib.reload(nw);

## 3. Training data

In [ ]:
# Download background noise, room echoes and the shared negative data (about 20 GB, 20-40 minutes).
import numpy as np, scipy.io.wavfile, datasets
from tqdm.auto import tqdm

def save16k(folder, name, array):
    Path(folder).mkdir(exist_ok=True)
    scipy.io.wavfile.write(os.path.join(folder, name), 16000, (np.clip(array, -1, 1) * 32767).astype(np.int16))

# Room impulse responses (MIT), used to add echo to the clips
# (a ".done" file marks a finished step, so a run cut off half-way starts that step again)
if not Path("mit_rirs/.done").exists():
    for row in tqdm(datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True), desc="room echoes"):
        save16k("mit_rirs", row["audio"]["path"].split("/")[-1], row["audio"]["array"])
    Path("mit_rirs/.done").touch()

# AudioSet: everyday sounds as background noise
if not Path("audioset_16k/.done").exists():
    !mkdir -p audioset && wget -q -c -O audioset/bal_train09.tar https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar && tar -xf audioset/bal_train09.tar -C audioset
    flacs = [str(p) for p in Path("audioset/audio").glob("**/*.flac")]
    audioset = datasets.Dataset.from_dict({"audio": flacs}).cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset, desc="audioset"):
        save16k("audioset_16k", Path(row["audio"]["path"]).with_suffix(".wav").name, row["audio"]["array"])
    Path("audioset_16k/.done").touch()

# Free Music Archive: music as background noise (optional - training still works without it)
background = ["./audioset_16k"]
try:
    if not Path("fma/.done").exists():
        fma = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
        fma = iter(fma.cast_column("audio", datasets.Audio(sampling_rate=16000)))
        for _ in tqdm(range(MUSIC_HOURS * 3600 // 30), desc="music"):  # 30-second clips
            row = next(fma)
            save16k("fma", Path(row["audio"]["path"]).with_suffix(".wav").name, row["audio"]["array"])
        Path("fma/.done").touch()
    background.append("./fma")
except Exception as error:  # the dataset sometimes moves; music is a nice-to-have
    print("Skipping music background:", error)

# Pre-computed openWakeWord features: ~2,000 hours of speech/noise that must NOT trigger,
# plus the validation set used to estimate false triggers per hour
# (wget -c resumes a download that was cut off, and returns at once when it is complete)
for name in ["openwakeword_features_ACAV100M_2000_hrs_16bit.npy", "validation_set_features.npy"]:
    !wget -q -c https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/{name}
print("Data ready. Background folders:", background)

## 4. Your recordings (optional)

In [ ]:
# Upload your recordings (only when USE_REAL_RECORDINGS = True).
# On your computer: zip the folder data/wake_samples into wake_samples.zip first.
# Outside Colab, put wake_samples.zip next to this notebook instead.
samples_root = None
if USE_REAL_RECORDINGS:
    if not Path("wake_samples.zip").exists():
        from google.colab import files
        uploaded = files.upload()  # choose wake_samples.zip
        if "wake_samples.zip" not in uploaded:
            raise SystemExit("Please upload a file named wake_samples.zip")
    samples_root = nw.unpack_samples(Path("wake_samples.zip"), Path("wake_samples"))
    for folder in sorted(p for p in samples_root.iterdir() if p.is_dir()):
        print(f"{folder.name}: {len(list(folder.glob('*.wav')))} recordings")
else:
    print("Using synthetic voices only.")

## 5. Train

In [ ]:
# Train. Each model: generate clips -> add real recordings -> augment -> train (about 30-60 min each on a T4).
import yaml

base_config = yaml.safe_load(open("openwakeword/examples/custom_model.yml"))
results = {}
for name in MODELS:
    print("=" * 30, name, "=" * 30)
    results[name] = nw.train_model(
        nw.PRESETS[name],
        base_config=base_config,
        train_script=Path("openwakeword/openwakeword/train.py"),
        work_dir=Path("."),
        size=SIZE,
        samples_root=samples_root,
        overrides={
            "piper_sample_generator_path": "./piper-sample-generator",
            "output_dir": "./nemo_models",
            "rir_paths": ["./mit_rirs"],
            "background_paths": background,
            "background_paths_duplication_rate": [1] * len(background),
            "false_positive_validation_data_path": "validation_set_features.npy",
            "feature_data_files": {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"},
        },
    )
    print("Saved", results[name].model_path)

## 6. Check

In [ ]:
# Check the models on clips they did not learn from.
import random

for name, result in results.items():
    print("=" * 30, name, "=" * 30)
    negatives = sorted(Path("nemo_models", name, "negative_test").glob("*.wav"))
    negatives = random.Random(0).sample(negatives, min(100, len(negatives)))
    neg = nw.score_wavs(result.model_path, negatives)
    print(f"Look-alike words (should stay LOW): {sum(s >= 0.5 for s in neg.values())}/{len(neg)} reach 0.5")
    if result.held_out:
        real = nw.score_wavs(result.model_path, result.held_out)
        for clip, score in sorted(real.items()):
            print(f"  {score:.2f}  {clip}")
        print(f"Your held-back recordings (should be HIGH): {sum(s >= 0.5 for s in real.values())}/{len(real)} reach 0.5")
print("Pick listening.wake_threshold between the two groups; 0.5 is the default.")

## 7. Download

In [ ]:
# Download the models, then on your computer:
#   1. unzip into models/wakeword/  (models/wakeword/hai_nemo.onnx, models/wakeword/tolong.onnx)
#   2. config.yaml -> listening.wake_engine: "openwakeword"
#   3. python -m aiforus --check
import zipfile
with zipfile.ZipFile("nemo_wakeword_models.zip", "w") as z:
    for result in results.values():
        z.write(result.model_path, result.model_path.name)
try:
    from google.colab import files
    files.download("nemo_wakeword_models.zip")
except ImportError:  # running outside Colab (e.g. Jupyter in WSL2)
    print("Models zipped to", Path("nemo_wakeword_models.zip").resolve())